In [1]:
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import random
from itertools import product
import optuna
import numpy as np
from scipy.stats import norm
import statsmodels.api as sm
import sys

def find_project_root(start: Path | None = None) -> Path:
    """Find repository root by walking up until .git or key project folders are found."""
    if start is None:
        start = Path.cwd().resolve()

    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists() or (
            (candidate / "Data").exists() and (candidate / "Empirical").exists()
        ):
            return candidate

    raise FileNotFoundError(
        f"Could not locate project root from {start}. Expected a parent containing .git or Data/ and Empirical/."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "Empirical" / "data"
SCRIPTS_DIR = PROJECT_ROOT / "Empirical" / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from stage1 import lasso_rolling_window, calculate_r_squared
from stage2 import estimate_kappa_curve_fit, compute_alm_returns, compute_stage2_r_squared
from grid_search import grid_search, estimate_single_config

In [2]:
# load feature matrix and response variable (absolute paths from project root)
feature_matrix = pd.read_csv(PROJECT_ROOT / "Data" / "clean_data" / "final_macro_topic_features.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv(PROJECT_ROOT / "Data" / "data_raw" / "market_return.csv", index_col=0, parse_dates=True)

In [3]:
def to_ar1_innovations(X: pd.DataFrame, min_obs: int = 30) -> pd.DataFrame:
    """Return AR(1) innovations (residuals) for each column of X."""
    X_innov = pd.DataFrame(index=X.index, columns=X.columns, dtype="float64")

    for col in X.columns:
        s = pd.to_numeric(X[col], errors="coerce")
        tmp = pd.DataFrame({"x": s, "x_lag1": s.shift(1)}).dropna()
        if len(tmp) < min_obs or tmp["x"].nunique() < 3 or tmp["x_lag1"].nunique() < 3:
            continue

        res = sm.OLS(tmp["x"], sm.add_constant(tmp["x_lag1"])).fit()
        X_innov.loc[tmp.index, col] = res.resid

    return X_innov

In [4]:
# set seed
random.seed(42)

# load feature matrix and response variable (absolute paths from project root)
feature_matrix = pd.read_csv(PROJECT_ROOT / "Data" / "clean_data" / "final_macro_topic_features.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv(PROJECT_ROOT / "Data" / "data_raw" / "market_return.csv", index_col=0, parse_dates=True)

# select a response variable (either marekt return or sp500 return)
y = response_variables['vwretx']  # or 'sprtrn' for SP500 returns or vwretx

# # transform returns to log
y = np.log(y+1)

# create feature matrix
X = feature_matrix.copy()

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# Final filtered dataframe
X = X[topic_cols]

# # usage
X = to_ar1_innovations(X)

# remove the first row wiht iloc
X = X.iloc[1:]

# ensure that all indices align
common_index = X.index.intersection(y.index)
X = X.loc[common_index]
y = y.loc[common_index]

In [5]:
from itertools import product
import pandas as pd
from joblib import Parallel, delayed
from tqdm import tqdm

def grid_search(
    X,
    y,
    param_grid,
    verbose=True,
    n_jobs=-1,
    backend="loky",
    prefer=None,
    return_details=True,
):
    """
    Parallel grid search.

    Parameters
    ----------
    return_details : bool, default True
        If False, skip collecting and concatenating window-level details
        (MUCH faster and lower memory usage).

    Returns
    -------
    results_df : pd.DataFrame
        Summary metrics for each configuration.
    coefficients_df : pd.DataFrame or None
        Detailed coefficients if return_details=True, else None.
    """

    combos = list(product(
        param_grid["window_sizes"],
        param_grid["n_lags"],
        param_grid["lambdas"],
    ))

    if verbose:
        print(f"Testing {len(combos)} configurations...")

    # -------------------------------------------------
    # Wrapper to prevent single failure from killing run
    # -------------------------------------------------
    def _safe_run(args):
        w, L, lam = args
        try:
            return estimate_single_config(X, y, w, L, lam)
        except Exception as e:
            if verbose:
                print(f"❌ Failed config (w={w}, L={L}, λ={lam}): {e}")
            return None

    iterator = tqdm(combos, desc="Grid search") if verbose else combos

    # -------------------------------------------------
    # Parallel execution
    # -------------------------------------------------
    results = Parallel(n_jobs=n_jobs, backend=backend, prefer=prefer)(
        delayed(_safe_run)(args) for args in iterator
    )

    # -------------------------------------------------
    # Aggregate results
    # -------------------------------------------------
    summary_list = []
    details_list = [] if return_details else None

    for res in results:
        if res is None:
            continue

        summary_list.append(res.get("summary", {}))

        if return_details:
            det = res.get("details", None)
            if det is not None and not det.empty:
                details_list.append(det)

    # -------------------------------------------------
    # Build summary DataFrame
    # -------------------------------------------------
    results_df = pd.DataFrame(summary_list)

    if not results_df.empty and "r2_oos_stage2" in results_df.columns:
        results_df = results_df.sort_values(
            "r2_oos_stage2", ascending=False
        ).reset_index(drop=True)

    # -------------------------------------------------
    # Build details DataFrame (optional)
    # -------------------------------------------------
    coefficients_df = None
    if return_details and details_list:
        coefficients_df = pd.concat(details_list, ignore_index=True)

    # -------------------------------------------------
    # Reporting
    # -------------------------------------------------
    if verbose:
        print("\n" + "=" * 80)
        print("GRID SEARCH COMPLETE")
        print("=" * 80)

        if "kappa" in results_df.columns:
            n_failed = results_df["kappa"].isna().sum()
            if n_failed > 0:
                print(f"⚠️  {n_failed}/{len(results_df)} configurations failed")

    return results_df, coefficients_df


## Bayesian Optimization with Optuna

> Alternative smooth objective (equal weights):

For each candidate `lambda` (with fixed `window_size=30`, `n_lags=1`), we run stage 1 and stage 2 and compute a weighted score based on four components:

1. `stage1_r2_score`: nonlinear bump around `r2_insample_stage1 = 0.10` (reward positive but not too large values).
2. `stage2_r2_score`: smooth increasing transform of `r2_insample_stage2`.
3. `kappa_score`: smooth increasing transform of `kappa` (penalizes negative/near-zero values).
4. `kappa_tstat_score`: smooth gate for `kappa_tstat > 1.9` with an upper-tail penalty to avoid numerical-explosion-like values.

The objective is the equal-weight average:

`objective = 0.25 * (stage1_r2_score + stage2_r2_score + kappa_score + kappa_tstat_score)`

In [20]:
import numpy as np
import optuna
from optuna.samplers import TPESampler

def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def objective_legacy(trial):
    """Original objective kept for comparison."""
    n_lags = 1
    window_size = 30
    lam = trial.suggest_float("lambda", 1e-7, 1e-3, log=True)

    res = estimate_single_config(
        X, y, window_size, n_lags, lam,
        standardize=True, verbose=False, return_details=False
    )
    summary = (res or {}).get("summary", {})

    r2_2 = float(summary.get("r2_insample_stage2", np.nan))
    kappa = float(summary.get("kappa", np.nan))
    r2_1 = float(summary.get("r2_insample_stage1", np.nan))
    kappa_tstat = float(summary.get("kappa_tstat", np.nan))

    if not np.isfinite(r2_2):
        raise optuna.TrialPruned()

    positive_r2_1 = r2_1 > 0
    kappa_significant = kappa_tstat > 1.96

    trial.set_user_attr("r2_stage2", r2_2)
    trial.set_user_attr("r2_stage1", r2_1)
    trial.set_user_attr("kappa", kappa)
    trial.set_user_attr("kappa_tstat", kappa_tstat)

    return r2_2 * kappa * positive_r2_1 * kappa_significant

def objective_weighted_smooth(trial):
    """
    Smooth equal-weight objective on four targets:
      - Stage 1 R^2 (nonlinear target around 0.10)
      - Stage 2 R^2 (monotone increasing, bounded)
      - Kappa (monotone increasing, bounded)
      - Kappa t-stat (>1.9 preferred, very large values penalized)
    """
    n_lags = 1
    window_size = 30
    lam = trial.suggest_float("lambda", 1e-7, 1e-3, log=True)

    res = estimate_single_config(
        X, y, window_size, n_lags, lam,
        standardize=True, verbose=False, return_details=False
    )
    summary = (res or {}).get("summary", {})

    r2_1 = float(summary.get("r2_insample_stage1", np.nan))
    r2_2 = float(summary.get("r2_insample_stage2", np.nan))
    kappa = float(summary.get("kappa", np.nan))
    kappa_tstat = float(summary.get("kappa_tstat", np.nan))

    metrics = np.array([r2_1, r2_2, kappa, kappa_tstat], dtype=float)
    if not np.all(np.isfinite(metrics)):
        raise optuna.TrialPruned()

    # 1) Stage-1 R^2 bump: maximum at 0.10, penalize both negative and too-large values.
    stage1_target = 0.10
    stage1_bandwidth = 0.06
    stage1_r2_score = np.exp(-((r2_1 - stage1_target) / stage1_bandwidth) ** 2)

    # 2) Stage-2 R^2: smooth monotone map to [0, 1].
    stage2_r2_score = _sigmoid(r2_2 / 0.02)

    # 3) Kappa: smooth monotone map to [0, 1], discourages non-positive values.
    kappa_score = _sigmoid(kappa / 0.02)

    # 4) Kappa t-stat: reward values above 1.9 but penalize extreme upper tail.
    t_threshold = 1.9
    t_upper_soft = 8.0
    t_rise = _sigmoid((kappa_tstat - t_threshold) / 0.35)
    t_fall = _sigmoid((t_upper_soft - kappa_tstat) / 1.2)
    kappa_tstat_score = t_rise * t_fall

    #weighted average across the four normalized scores.
    objective_value = (
        0.9 *stage1_r2_score + 0.01*stage2_r2_score + 0.08*kappa_score + 0.01*kappa_tstat_score
    )

    #try with geometric mean instead of arithmetic mean
    #objective_value = (stage1_r2_score * stage2_r2_score * kappa_score * kappa_tstat_score) ** 0.25

    trial.set_user_attr("r2_stage1", r2_1)
    trial.set_user_attr("r2_stage2", r2_2)
    trial.set_user_attr("kappa", kappa)
    trial.set_user_attr("kappa_tstat", kappa_tstat)
    trial.set_user_attr("score_stage1_r2", float(stage1_r2_score))
    trial.set_user_attr("score_stage2_r2", float(stage2_r2_score))
    trial.set_user_attr("score_kappa", float(kappa_score))
    trial.set_user_attr("score_kappa_tstat", float(kappa_tstat_score))

    return float(objective_value)

sampler = TPESampler(
    seed=42,
    multivariate=True,
    n_startup_trials=50,
    n_ei_candidates=64
)

study1 = optuna.create_study(direction="maximize", sampler=sampler)
study1.optimize(objective_weighted_smooth, n_trials=30, n_jobs=6, gc_after_trial=True)

[I 2026-03-19 16:05:24,672] A new study created in memory with name: no-name-c341d52c-f3d2-48f3-bcc7-7b5b75579c39
[I 2026-03-19 16:05:43,436] Trial 4 finished with value: 0.06566920280927654 and parameters: {'lambda': 7.269826789874461e-06}. Best is trial 4 with value: 0.06566920280927654.
[I 2026-03-19 16:05:43,977] Trial 0 finished with value: 0.06503300122373491 and parameters: {'lambda': 6.181150899499709e-06}. Best is trial 4 with value: 0.06566920280927654.
[I 2026-03-19 16:05:44,187] Trial 3 finished with value: 0.06733169557296041 and parameters: {'lambda': 5.610275539991452e-06}. Best is trial 3 with value: 0.06733169557296041.
[I 2026-03-19 16:05:46,974] Trial 2 finished with value: 0.08925510555948685 and parameters: {'lambda': 2.126666876224683e-06}. Best is trial 2 with value: 0.08925510555948685.
[I 2026-03-19 16:05:52,640] Trial 5 finished with value: 0.07979435555492485 and parameters: {'lambda': 2.821398167074572e-07}. Best is trial 2 with value: 0.08925510555948685.
[

In [21]:
trials_df = study1.trials_dataframe(
    attrs=("number", "value", "state", "params", "user_attrs", "system_attrs")
)

# save to csv
trials_df.to_csv("optuna_study_trials.csv", index=False)

In [8]:
# print(trials_df.columns)

In [22]:
# sort from highest composite objective score to lowest
df = trials_df.sort_values("value", ascending=False)

param_cols = [c for c in df.columns if c.startswith("params_")]

for _, row in df.iterrows():
    params = {c.replace("params_", ""): row[c] for c in param_cols}

    print(
        f"Trial {int(row['number'])}: "
        f"score={row['value']:.6f}, "
        f"Params={params}, "
        f"kappa={row.get('user_attrs_kappa', float('nan')):.6g}, "
        f"t={row.get('user_attrs_kappa_tstat', float('nan')):.2f}, "
        f"R2_stage1={row.get('user_attrs_r2_stage1', float('nan')):.6f}, "
        f"R2_stage2={row.get('user_attrs_r2_stage2', float('nan')):.6f}, "
        f"S1={row.get('user_attrs_score_stage1_r2', float('nan')):.4f}, "
        f"S2={row.get('user_attrs_score_stage2_r2', float('nan')):.4f}, "
        f"Sk={row.get('user_attrs_score_kappa', float('nan')):.4f}, "
        f"St={row.get('user_attrs_score_kappa_tstat', float('nan')):.4f}"
    )

Trial 17: score=0.091000, Params={'lambda': 9.14980476814398e-07}, kappa=0.0591198, t=2.98, R2_stage1=0.973767, R2_stage2=0.004269, S1=0.0000, S2=0.5532, Sk=0.9505, St=0.9424
Trial 13: score=0.090909, Params={'lambda': 8.803055410588197e-07}, kappa=0.0587506, t=2.97, R2_stage1=0.973767, R2_stage2=0.004230, S1=0.0000, S2=0.5527, Sk=0.9497, St=0.9408
Trial 24: score=0.090156, Params={'lambda': 1.1656286268011966e-06}, kappa=0.0564902, t=2.80, R2_stage1=0.973767, R2_stage2=0.003775, S1=0.0000, S2=0.5471, Sk=0.9440, St=0.9167
Trial 18: score=0.089882, Params={'lambda': 1.302855960487345e-06}, kappa=0.0557409, t=2.75, R2_stage1=0.973767, R2_stage2=0.003645, S1=0.0000, S2=0.5454, Sk=0.9420, St=0.9070
Trial 25: score=0.089407, Params={'lambda': 1.9828180796374384e-06}, kappa=0.0544385, t=2.68, R2_stage1=0.973766, R2_stage2=0.003466, S1=0.0000, S2=0.5432, Sk=0.9383, St=0.8911
Trial 2: score=0.089255, Params={'lambda': 2.126666876224683e-06}, kappa=0.0540084, t=2.66, R2_stage1=0.973766, R2_stag

In [10]:
#check the best trial
best_trial = study1.best_trial
print(f"Best trial: {best_trial.number}, Value: {best_trial.value}, Params: {best_trial.params}")


Best trial: 18, Value: 1.4824199431779575e-17, Params: {'lambda': 0.0007803586260941518}
